In [17]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder

In [20]:
df_train = pd.read_csv('train_final_processed.csv')
df_test = pd.read_csv('test_final_processed.csv')
print(f"Размер тренировочных данных: {df_train.shape}")
print(f"Размер тестовых данных: {df_test.shape}")

Размер тренировочных данных: (307511, 571)
Размер тестовых данных: (48744, 570)


In [21]:
X = df_train.drop(['sk_id_curr', 'target'], axis=1)
y = df_train['target']

In [22]:
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

In [28]:
if y.dtype == 'object' or len(y.unique()) > 2:
    print("Предупреждение: target содержит некорректные значения. Преобразуем...")
    y_train_full = y.map({'0': 0, '1': 1, 'Unaccompanied': 1}).fillna(0).astype(int)
    print("Новые уникальные значения в target:", y_train_full.unique())

In [29]:
common_cols = X.columns.intersection(df_test.drop(['sk_id_curr'], axis=1).columns)
print(f"Общие столбцы: {common_cols.tolist()}")
if len(common_cols) < 10:
    print("Предупреждение: Очень мало общих столбцов! Проверьте предобработку данных.")
X_train_full = X[common_cols]
df_test_processed = df_test[common_cols]

Общие столбцы: ['name_contract_type', 'code_gender', 'flag_own_car', 'flag_own_realty', 'cnt_children', 'amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price', 'name_type_suite', 'name_income_type', 'name_education_type', 'name_family_status', 'name_housing_type', 'region_population_relative', 'days_birth', 'days_employed', 'days_registration', 'days_id_publish', 'own_car_age', 'flag_mobil', 'flag_emp_phone', 'flag_work_phone', 'flag_cont_mobile', 'flag_phone', 'flag_email', 'occupation_type', 'cnt_fam_members', 'region_rating_client', 'region_rating_client_w_city', 'weekday_appr_process_start', 'hour_appr_process_start', 'reg_region_not_live_region', 'reg_region_not_work_region', 'live_region_not_work_region', 'reg_city_not_live_city', 'reg_city_not_work_city', 'live_city_not_work_city', 'organization_type', 'ext_source_1', 'ext_source_2', 'ext_source_3', 'apartments_avg', 'basementarea_avg', 'years_beginexpluatation_avg', 'years_build_avg', 'commonarea_avg', 'elevators_av

In [30]:
cat_cols = [col for col in X_train_full.select_dtypes(include=['object']).columns if col in common_cols]
print(f"Категориальные столбцы: {cat_cols}")
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X_train_full[col] = X_train_full[col].astype(str).fillna('missing')
    X_train_full[col] = le.fit_transform(X_train_full[col])
    df_test_processed[col] = df_test_processed[col].astype(str).fillna('missing')
    df_test_processed[col] = le.transform(df_test_processed[col])

Категориальные столбцы: ['name_income_type', 'name_education_type', 'name_family_status', 'name_housing_type', 'occupation_type', 'weekday_appr_process_start', 'organization_type', 'fondkapremont_mode', 'housetype_mode', 'wallsmaterial_mode', 'emergencystate_mode']


C:\Users\klmv0\AppData\Local\Temp\ipykernel_34168\3166392836.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_processed[col] = df_test_processed[col].astype(str).fillna('missing')
C:\Users\klmv0\AppData\Local\Temp\ipykernel_34168\3166392836.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_processed[col] = le.transform(df_test_processed[col])
C:\Users\klmv0\AppData\Local\Temp\ipykernel_34168\3166392836.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from 

In [31]:
for col in X_train_full.columns:
    if X_train_full[col].dtype in ['float64', 'int64']:
        X_train_full[col].fillna(X_train_full[col].median(), inplace=True)
        df_test_processed[col].fillna(X_train_full[col].median(), inplace=True)

C:\Users\klmv0\AppData\Local\Temp\ipykernel_34168\2300449642.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_processed[col].fillna(X_train_full[col].median(), inplace=True)
C:\Users\klmv0\AppData\Local\Temp\ipykernel_34168\2300449642.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_processed[col].fillna(X_train_full[col].median(), inplace=True)
C:\Users\klmv0\AppData\Local\Temp\ipykernel_34168\2300449642.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/ind

In [32]:
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

In [34]:
cat_model_optimal = CatBoostClassifier(
    iterations=1000,     
    learning_rate=0.03,       
    depth=7,          
    auto_class_weights='SqrtBalanced',
    random_state=42,
    l2_leaf_reg=2,  
    eval_metric='AUC',
    loss_function='Logloss',
    verbose=100)

cat_model_optimal.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_cols)

0:	test: 0.6946247	best: 0.6946247 (0)	total: 4.17s	remaining: 1h 9m 30s
100:	test: 0.7634826	best: 0.7634826 (100)	total: 2m 22s	remaining: 21m 5s
200:	test: 0.7740676	best: 0.7740676 (200)	total: 4m 16s	remaining: 16m 58s
300:	test: 0.7783514	best: 0.7783514 (300)	total: 6m	remaining: 13m 57s
400:	test: 0.7812114	best: 0.7812114 (400)	total: 7m 42s	remaining: 11m 30s
500:	test: 0.7835584	best: 0.7835756 (498)	total: 9m 24s	remaining: 9m 21s
600:	test: 0.7853512	best: 0.7853680 (598)	total: 11m 11s	remaining: 7m 25s
700:	test: 0.7861257	best: 0.7861257 (700)	total: 12m 51s	remaining: 5m 28s
800:	test: 0.7869609	best: 0.7869811 (798)	total: 14m 29s	remaining: 3m 36s
900:	test: 0.7874970	best: 0.7874989 (897)	total: 16m 12s	remaining: 1m 46s
999:	test: 0.7879108	best: 0.7879174 (998)	total: 17m 49s	remaining: 0us

bestTest = 0.7879174242
bestIteration = 998

Shrink model to first 999 iterations.


In [35]:
y_val_pred_proba = cat_model_optimal.predict_proba(X_val)[:, 1]
val_metrics = {
    'ROC-AUC': roc_auc_score(y_val, y_val_pred_proba),
    'Accuracy': accuracy_score(y_val, y_val_pred_proba > 0.5),
    'Precision': precision_score(y_val, y_val_pred_proba > 0.5),
    'Recall': recall_score(y_val, y_val_pred_proba > 0.5),
    'F1-Score': f1_score(y_val, y_val_pred_proba > 0.5)
}
for metric, value in val_metrics.items():
    print(f'Validation {metric}: {value:.4f}')

Validation ROC-AUC: 0.7879
Validation Accuracy: 0.9040
Validation Precision: 0.3670
Validation Recall: 0.2614
Validation F1-Score: 0.3053
